[//]: # (cr:doc name='section' id=5dfd2557)


[//]: # (cr:doc name='chapter_5_relationship_analysis' id=f85cadd3)
# Chapter 5: Relationship Analysis

**Purpose:** Explore feature correlations, relationships with the target, and identify predictive signals.

**What you'll learn:**
- How to interpret correlation matrices and identify multicollinearity
- How to visualize feature distributions by target class
- How to identify which features have the strongest relationship with retention
- How to analyze categorical features for predictive power

**Outputs:**
- Correlation heatmap with multicollinearity detection
- Feature distributions by retention status (box plots)
- Retention rates by categorical features
- Feature-target correlation rankings

---

## Understanding Feature Relationships

| Analysis | What It Tells You | Action |
|----------|------------------|--------|
| **High Correlation** (r > 0.7) | Features carry redundant information | Consider removing one |
| **Target Correlation** | Feature's predictive power | Prioritize high-correlation features |
| **Class Separation** | How different retained vs churned look | Good separation = good predictor |
| **Categorical Rates** | Retention varies by category | Use for segmentation and encoding |

[//]: # (cr:doc name='5_1_setup' id=7576bf8b)
## 5.1 Setup

In [1]:
# @cr:code name='init_progress' id=6850ee7b
from customer_retention.analysis.notebook_progress import accept_workflow_params, track_and_export_previous

accept_workflow_params()
track_and_export_previous("05_relationship_analysis.ipynb")

import numpy as np
import plotly.graph_objects as go
import yaml
from plotly.subplots import make_subplots

from customer_retention.analysis.auto_explorer import ExplorationFindings, ExplorationManager, RecommendationRegistry
from customer_retention.analysis.visualization import ChartBuilder, display_figure, display_table
from customer_retention.core.compat import batched_corr_matrix, bulk_effect_sizes, native_pd, safe_sample
from customer_retention.core.config.column_config import ColumnType
from customer_retention.core.config.experiments import FINDINGS_DIR  # noqa: F401
from customer_retention.core.utils.leakage import detect_leaking_features
from customer_retention.stages.profiling import RecommendationCategory, RelationshipRecommender

In [2]:
# @cr:code name='load_findings' id=09b14f1e
from customer_retention.analysis.auto_explorer import load_notebook_findings

FINDINGS_PATH, _namespace, dataset_name = load_notebook_findings(
    "05_relationship_analysis.ipynb", prefer_merged=True
)
print(f"Using: {FINDINGS_PATH}")

RECOMMENDATIONS_PATH = str(_namespace.merged_recommendations_path)

findings = ExplorationFindings.load(FINDINGS_PATH)

from customer_retention.analysis.auto_explorer.active_dataset_store import require_silver_merged
from customer_retention.stages.temporal import TEMPORAL_METADATA_COLS

df = require_silver_merged(_namespace)
data_source = "silver_merged"

_df_cols = set(df.columns)
findings.columns = {k: v for k, v in findings.columns.items() if k in _df_cols}
if findings.target_column and findings.target_column not in _df_cols:
    findings.target_column = None

charts = ChartBuilder()

if _namespace.merged_recommendations_path.exists():
    with open(str(_namespace.merged_recommendations_path), "r") as f:
        registry = RecommendationRegistry.from_dict(yaml.safe_load(f))
    print(f"Loaded existing recommendations: {len(registry.all_recommendations)} total")
else:
    registry = RecommendationRegistry()
    registry.init_bronze(findings.source_path)
    _entity_col = (findings.time_series_metadata.entity_column
                   if findings.time_series_metadata else None)
    registry.init_silver(_entity_col or "entity_id")
    registry.init_gold(findings.target_column or "target")
    print("Initialized new recommendation registry")

print(f"\nLoaded {len(df):,} rows from: {data_source}")

Using: /Users/Vital/python/CustomerRetention/experiments/runs/3set-1e30406f/merged/silver_merged_findings.yaml


Loaded existing recommendations: 452 total

Loaded 193,000 rows from: silver_merged


[//]: # (cr:doc name='5_1b_leakage_exclusion_gate' id=52692d66)
## 5.1b Leakage Exclusion Gate

Features that leak target information are automatically detected and removed before relationship analysis.
Add column names to `EXCLUDE_LEAKING_FEATURES` to manually exclude additional features you suspect of leakage.

In [3]:
# @cr:code name='check_leaking_features' id=ecad955f
EXCLUDE_LEAKING_FEATURES = []

_check_cols = [
    name for name, col in findings.columns.items()
    if col.inferred_type in [ColumnType.NUMERIC_CONTINUOUS, ColumnType.NUMERIC_DISCRETE, ColumnType.BINARY]
    and name != findings.target_column
    and name not in TEMPORAL_METADATA_COLS
]

_auto_leakers = detect_leaking_features(df, _check_cols, findings.target_column)
_all_excluded = sorted(set(_auto_leakers) | set(EXCLUDE_LEAKING_FEATURES))

if _all_excluded:
    for _col in _all_excluded:
        findings.columns.pop(_col, None)
    df = df.drop(columns=[c for c in _all_excluded if c in df.columns])
    findings.excluded_leaking_features = _all_excluded

    _auto_only = [c for c in _auto_leakers if c not in EXCLUDE_LEAKING_FEATURES]
    _manual_only = [c for c in EXCLUDE_LEAKING_FEATURES if c not in _auto_leakers]
    print(f"Excluded {len(_all_excluded)} leaking feature(s):")
    if _auto_only:
        print(f"  Auto-detected: {', '.join(_auto_only)}")
    if _manual_only:
        print(f"  Manual: {', '.join(_manual_only)}")
    _both = [c for c in _all_excluded if c in _auto_leakers and c in EXCLUDE_LEAKING_FEATURES]
    if _both:
        print(f"  Both: {', '.join(_both)}")
else:
    print("No leaking features detected.")

/Users/Vital/python/CustomerRetention/src/customer_retention/core/utils/leakage.py:92: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  null_df[alias] = df[col].isna().astype(float)
/Users/Vital/python/CustomerRetention/src/customer_retention/core/utils/leakage.py:92: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  null_df[alias] = df[col].isna().astype(float)
/Users/Vital/python/CustomerRetention/src/customer_retention/core/utils/leakage.py:92: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of callin

/Users/Vital/python/CustomerRetention/src/customer_retention/core/utils/leakage.py:92: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  null_df[alias] = df[col].isna().astype(float)
/Users/Vital/python/CustomerRetention/src/customer_retention/core/utils/leakage.py:92: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  null_df[alias] = df[col].isna().astype(float)
/Users/Vital/python/CustomerRetention/src/customer_retention/core/utils/leakage.py:92: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of callin

/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-p

/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-p

/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-p

/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-p

/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-p

/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-p

No leaking features detected.


/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/Vital/python/CustomerRetention/.venv/lib/python3.12/site-p

[//]: # (cr:doc name='5_2_numeric_correlation_matrix' id=245cd8a1)
## 5.2 Numeric Correlation Matrix

**📖 How to Read the Heatmap:**
- **Red (+1)**: Perfect positive correlation - features move together
- **Blue (-1)**: Perfect negative correlation - features move opposite
- **White (0)**: No linear relationship

**⚠️ Multicollinearity Warning:**
- Pairs with |r| > 0.7 may cause issues in linear models
- Consider removing one feature from highly correlated pairs
- Tree-based models are more robust to multicollinearity

In [4]:
# @cr:code name='compute_correlations' id=1fa22c14
numeric_cols = [
    name for name, col in findings.columns.items()
    if col.inferred_type in [ColumnType.NUMERIC_CONTINUOUS, ColumnType.NUMERIC_DISCRETE, ColumnType.TARGET]
    and name not in TEMPORAL_METADATA_COLS
]

corr_matrix = None
if len(numeric_cols) >= 2:
    corr_matrix = batched_corr_matrix(df, numeric_cols)
    fig = charts.heatmap(
        corr_matrix.to_numpy(),
        x_labels=numeric_cols,
        y_labels=numeric_cols,
        title="Numeric Correlation Matrix"
    )
    display_figure(fig)
else:
    print("Not enough numeric columns for correlation analysis.")

[//]: # (cr:doc name='5_3_high_correlation_pairs' id=a08cb7b9)
## 5.3 High Correlation Pairs

In [5]:
# @cr:code name='find_high_correlations' id=03e56f1f
high_corr_threshold = 0.7
high_corr_pairs = []

if corr_matrix is not None and len(numeric_cols) >= 2:
    for i in range(len(numeric_cols)):
        for j in range(i+1, len(numeric_cols)):
            corr_val = corr_matrix.iloc[i, j]
            if abs(corr_val) >= high_corr_threshold:
                high_corr_pairs.append({
                    "Column 1": numeric_cols[i],
                    "Column 2": numeric_cols[j],
                    "Correlation": f"{corr_val:.3f}"
                })

if high_corr_pairs:
    print(f"High Correlation Pairs (|r| >= {high_corr_threshold}):")
    display_table(native_pd.DataFrame(high_corr_pairs))
    print("\nConsider removing one of each pair to reduce multicollinearity.")
else:
    print("No high correlation pairs detected.")

High Correlation Pairs (|r| >= 0.7):


Column 1,Column 2,Correlation
event_count_7d,event_count_14d,0.995
event_count_7d,event_count_30d,0.965
event_count_7d,edi_transaction_type_sum_7d,1.000
event_count_7d,edi_transaction_type_count_7d,1.000
event_count_7d,amount_sum_7d,0.994
event_count_7d,amount_max_7d,0.960
event_count_7d,amount_count_7d,1.000
event_count_7d,file_size_kb_sum_7d,0.994
event_count_7d,file_size_kb_max_7d,0.722
event_count_7d,file_size_kb_count_7d,1.000



Consider removing one of each pair to reduce multicollinearity.


[//]: # (cr:doc name='5_4_feature_distributions_by_retention_status' id=817a8b34)
## 5.4 Feature Distributions by Retention Status

**📖 How to Interpret Box Plots:**
- **Box** = Middle 50% of data (IQR)
- **Line inside box** = Median
- **Whiskers** = 1.5 × IQR from box edges
- **Points outside** = Outliers

**⚠️ What Makes a Good Predictor:**
- **Clear separation** between retained (green) and churned (red) boxes
- **Different medians** = Feature values differ between classes
- **Minimal overlap** = Easier to distinguish classes

In [6]:
# @cr:code name='plot_feature_distributions' id=ae2926c9
# Feature Distributions by Retention Status
_effect_sizes_result = None

if findings.target_column and findings.target_column in df.columns:
    target = findings.target_column

    feature_cols = [
        name for name, col in findings.columns.items()
        if col.inferred_type in [ColumnType.NUMERIC_CONTINUOUS, ColumnType.NUMERIC_DISCRETE]
        and name != target
        and name not in TEMPORAL_METADATA_COLS
    ]

    if feature_cols:
        print("=" * 80)
        print(f"FEATURE DISTRIBUTIONS BY TARGET: {target}")
        print("=" * 80)

        # Bulk compute effect sizes and per-class stats (2 Spark agg calls instead of 16N)
        _effect_sizes_result = bulk_effect_sizes(df, feature_cols, target)

        # Build summary table from bulk class_stats
        summary_by_target = []
        for col in feature_cols:
            stats = _effect_sizes_result.class_stats.get(col)
            if not stats:
                continue
            for cls, label in [(0, "Churned"), (1, "Retained")]:
                if stats[f"count_{cls}"] > 0:
                    summary_by_target.append({
                        "Feature": col,
                        "Group": label,
                        "Count": stats[f"count_{cls}"],
                        "Mean": stats[f"mean_{cls}"],
                        "Median": stats[f"median_{cls}"],
                        "Std": stats[f"std_{cls}"],
                    })

        if summary_by_target:
            summary_df = native_pd.DataFrame(summary_by_target)

            # Display summary table
            print("\n📊 Summary Statistics by Retention Status:")
            display_summary = summary_df.pivot(index="Feature", columns="Group", values=["Mean", "Median"])
            display_summary.columns = [f"{stat} ({group})" for stat, group in display_summary.columns]
            display_table(display_summary.round(3))

        # Display effect sizes from bulk result
        print("\n📈 Feature Importance Indicators (Effect Size - Cohen's d):")
        print("-" * 70)
        effect_sizes = []
        for col in feature_cols:
            d = _effect_sizes_result.effect_sizes.get(col)
            if d is None:
                continue

            abs_d = abs(d)
            if abs_d >= 0.8:
                interpretation = "Large effect"
                emoji = "🔴"
            elif abs_d >= 0.5:
                interpretation = "Medium effect"
                emoji = "🟡"
            elif abs_d >= 0.2:
                interpretation = "Small effect"
                emoji = "🟢"
            else:
                interpretation = "Negligible"
                emoji = "⚪"

            effect_sizes.append({
                "feature": col,
                "cohens_d": d,
                "abs_d": abs_d,
                "interpretation": interpretation
            })

            direction = "↑ Higher in retained" if d > 0 else "↓ Lower in retained"
            print(f"  {emoji} {col}: d={d:+.3f} ({interpretation}) {direction}")

        # Sort by effect size for identifying important features
        if effect_sizes:
            effect_df = native_pd.DataFrame(effect_sizes).sort_values("abs_d", ascending=False)
            important_features = effect_df[effect_df["abs_d"] >= 0.2]["feature"].tolist()
            if important_features:
                print(f"\n⭐ Features with notable effect (|d| ≥ 0.2): {', '.join(important_features)}")
        else:
            print("  No effect sizes could be calculated (insufficient data in one or both groups)")
    else:
        print("No numeric feature columns found for distribution analysis.")
else:
    print("Target column not available.")

FEATURE DISTRIBUTIONS BY TARGET: churned



📊 Summary Statistics by Retention Status:


Mean (Churned),Mean (Retained),Median (Churned),Median (Retained)
340.226,338.380,352.000,350.500
7.651,-120.338,24.406,5.891
25916.175,26010.804,6081.895,5419.875
0.006,-0.053,-0.302,-0.301
0.058,0.000,0.000,0.000
6.219,0.000,0.000,0.000
0.183,0.000,0.000,0.000
0.028,0.000,0.000,0.000
1.634,0.000,0.000,0.000
124.664,126.280,53.000,48.500



📈 Feature Importance Indicators (Effect Size - Cohen's d):
----------------------------------------------------------------------
  ⚪ event_count_7d: d=-0.048 (Negligible) ↓ Lower in retained
  ⚪ event_count_14d: d=-0.055 (Negligible) ↓ Lower in retained
  ⚪ event_count_30d: d=-0.079 (Negligible) ↓ Lower in retained
  ⚪ event_count_90d: d=-0.155 (Negligible) ↓ Lower in retained
  🟢 event_count_180d: d=-0.233 (Small effect) ↓ Lower in retained
  ⚪ event_count_all_time: d=+0.009 (Negligible) ↑ Higher in retained
  ⚪ edi_transaction_type_sum_7d: d=-0.048 (Negligible) ↓ Lower in retained
  ⚪ edi_transaction_type_mean_7d: d=+0.000 (Negligible) ↓ Lower in retained
  ⚪ edi_transaction_type_max_7d: d=+0.000 (Negligible) ↓ Lower in retained
  ⚪ edi_transaction_type_count_7d: d=-0.048 (Negligible) ↓ Lower in retained
  ⚪ amount_sum_7d: d=-0.041 (Negligible) ↓ Lower in retained
  ⚪ amount_mean_7d: d=+0.000 (Negligible) ↓ Lower in retained
  ⚪ amount_max_7d: d=+0.000 (Negligible) ↓ Lower in retai

[//]: # (cr:doc name='interpreting_effect_sizes_cohen_s_d' id=0157c38f)
### Interpreting Effect Sizes (Cohen's d)

| Effect Size | Interpretation | What It Means for Modeling |
|-------------|----------------|---------------------------|
| \|d\| ≥ 0.8 | Large | Strong discriminator - prioritize this feature |
| \|d\| = 0.5-0.8 | Medium | Useful predictor - include in model |
| \|d\| = 0.2-0.5 | Small | Weak but may help in combination with others |
| \|d\| < 0.2 | Negligible | Limited predictive value alone |

**🎯 Actionable Insights:**
- **Features with large effects** are your best predictors - ensure they're included in your model
- **Direction matters**: "Higher in retained" means customers with high values tend to stay; use this for threshold-based business rules
- **Features with small/negligible effects** may still be useful in combination or as interaction terms

**⚠️ Cautions:**
- Effect size assumes roughly normal distributions - check skewness in notebook 03
- Large effects could be due to confounding variables - validate with domain knowledge
- Correlation ≠ causation: high engagement may not *cause* retention

### Box Plot Visualization

**📈 How to Read the Box Plots Below:**
- **Well-separated boxes** (little/no overlap) → Feature clearly distinguishes retained vs churned
- **Different medians** (center lines at different heights) → Groups have different typical values
- **Many outliers in one group** → May indicate subpopulations worth investigating

In [7]:
# @cr:code name='compute_feature_importance' id=f6986cb3
if findings.target_column and findings.target_column in df.columns:
    target = findings.target_column

    feature_cols = [
        name for name, col in findings.columns.items()
        if col.inferred_type in [ColumnType.NUMERIC_CONTINUOUS, ColumnType.NUMERIC_DISCRETE]
        and name != target
        and name not in TEMPORAL_METADATA_COLS
    ]

    if feature_cols:
        n_features = min(len(feature_cols), 6)
        plot_cols = feature_cols[:n_features]

        # Collect box-plot columns + target in a single call
        _box_data = df[plot_cols + [target]].to_numpy() if hasattr(df, 'to_spark') else df[plot_cols + [target]].values
        _box_target = _box_data[:, -1]

        fig = make_subplots(
            rows=1, cols=n_features,
            subplot_titles=plot_cols,
            horizontal_spacing=0.05
        )

        for i, col in enumerate(plot_cols):
            col_num = i + 1
            col_vals = _box_data[:, i]

            retained_mask = _box_target == 1
            retained_vals = col_vals[retained_mask]
            retained_data = retained_vals[~np.isnan(retained_vals)]

            churned_mask = _box_target == 0
            churned_vals = col_vals[churned_mask]
            churned_data = churned_vals[~np.isnan(churned_vals)]

            fig.add_trace(
                go.Box(
                    y=retained_data,
                    name='Retained',
                    fillcolor='rgba(46, 204, 113, 0.7)',
                    line=dict(color='#1e8449', width=2),
                    marker=dict(color='rgba(46, 204, 113, 0.5)', size=5,
                                line=dict(color='#1e8449', width=1)),
                    boxpoints='outliers', width=0.35,
                    showlegend=(i == 0), legendgroup='retained', offsetgroup='retained'
                ),
                row=1, col=col_num
            )

            fig.add_trace(
                go.Box(
                    y=churned_data,
                    name='Churned',
                    fillcolor='rgba(231, 76, 60, 0.7)',
                    line=dict(color='#922b21', width=2),
                    marker=dict(color='rgba(231, 76, 60, 0.5)', size=5,
                                line=dict(color='#922b21', width=1)),
                    boxpoints='outliers', width=0.35,
                    showlegend=(i == 0), legendgroup='churned', offsetgroup='churned'
                ),
                row=1, col=col_num
            )

        fig.update_layout(
            height=450,
            title_text="Feature Distributions: Retained (Green) vs Churned (Red)",
            template='plotly_white', showlegend=True,
            legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5),
            boxmode='group', boxgap=0.3, boxgroupgap=0.1
        )
        fig.update_xaxes(showticklabels=False)
        display_figure(fig)

        # Reuse bulk class_stats for mean comparison (no extra Spark jobs)
        print("\n📊 MEAN COMPARISON BY RETENTION STATUS:")
        print("-" * 70)
        for col in plot_cols:
            if _effect_sizes_result and col in _effect_sizes_result.class_stats:
                stats = _effect_sizes_result.class_stats[col]
                retained_mean = stats["mean_1"]
                churned_mean = stats["mean_0"]
            else:
                retained_mean = float(df[df[target] == 1][col].mean())
                churned_mean = float(df[df[target] == 0][col].mean())
            diff_pct = ((retained_mean - churned_mean) / churned_mean * 100) if churned_mean != 0 else 0
            print(f"  {col}:")
            print(f"     Retained: {retained_mean:.2f}  |  Churned: {churned_mean:.2f}  |  Diff: {diff_pct:+.1f}%")


📊 MEAN COMPARISON BY RETENTION STATUS:
----------------------------------------------------------------------
  event_count_7d:
     Retained: 0.00  |  Churned: 0.03  |  Diff: -100.0%
  event_count_14d:
     Retained: 0.00  |  Churned: 0.06  |  Diff: -100.0%
  event_count_30d:
     Retained: 0.00  |  Churned: 0.18  |  Diff: -100.0%
  event_count_90d:
     Retained: 0.00  |  Churned: 1.63  |  Diff: -100.0%
  event_count_180d:
     Retained: 0.00  |  Churned: 6.22  |  Diff: -100.0%
  event_count_all_time:
     Retained: 126.28  |  Churned: 124.66  |  Diff: +1.3%


[//]: # (cr:doc name='5_5_feature_target_correlations' id=763d38a5)
## 5.5 Feature-Target Correlations

Features ranked by absolute correlation with the target variable.

**📖 Interpretation:**
- **Positive correlation**: Higher values = more likely retained
- **Negative correlation**: Higher values = more likely churned
- **|r| > 0.3**: Moderately predictive
- **|r| > 0.5**: Strongly predictive

In [8]:
# @cr:code name='analyze_interactions' id=81ec4d57
if findings.target_column and findings.target_column in df.columns:
    target = findings.target_column
    feature_cols = [
        name for name, col in findings.columns.items()
        if col.inferred_type in [ColumnType.NUMERIC_CONTINUOUS, ColumnType.NUMERIC_DISCRETE]
        and name != target
        and name not in TEMPORAL_METADATA_COLS
    ]

    if feature_cols:
        # Reuse corr_matrix from 5.2 (already includes target via ColumnType.TARGET)
        if corr_matrix is not None and target in corr_matrix.index:
            target_corr_matrix = corr_matrix
        else:
            target_corr_matrix = batched_corr_matrix(df, feature_cols + [target])

        correlations = [
            {"Feature": col, "Correlation": target_corr_matrix.loc[col, target]}
            for col in feature_cols
            if col in target_corr_matrix.index
        ]

        corr_df = native_pd.DataFrame(correlations).sort_values("Correlation", key=abs, ascending=False)

        fig = charts.bar_chart(
            corr_df["Feature"].tolist(),
            corr_df["Correlation"].tolist(),
            title=f"Feature Correlations with {target}"
        )
        display_figure(fig)
else:
    print("Target column not available for correlation analysis.")

[//]: # (cr:doc name='5_6_categorical_feature_analysis' id=220b8f56)
## 5.6 Categorical Feature Analysis

Retention rates by category help identify which segments are at higher risk.

**📖 What to Look For:**
- Categories with **low retention rates** = high-risk segments for intervention
- **Large variation** across categories = strong predictive feature
- **Small categories** with extreme rates may be unreliable (small sample size)

**📊 Metrics Explained:**
- **Retention Rate**: % of customers in category who were retained
- **Lift**: How much better/worse than overall retention rate (>1 = better, <1 = worse)
- **Cramér's V**: Strength of association (0-1 scale, like correlation for categorical)

In [9]:
# @cr:code name='analyze_categorical_target' id=3e26270f
from customer_retention.stages.profiling import CategoricalTargetAnalyzer

if findings.target_column:
    target = findings.target_column
    overall_retention = df[target].mean()

    categorical_cols = [
        name for name, col in findings.columns.items()
        if col.inferred_type in [ColumnType.CATEGORICAL_NOMINAL, ColumnType.CATEGORICAL_ORDINAL]
        and name not in TEMPORAL_METADATA_COLS
    ]

    print("=" * 80)
    print("CATEGORICAL FEATURE ANALYSIS")
    print("=" * 80)
    print(f"Overall retention rate: {overall_retention:.1%}")

    if categorical_cols:
        # Use framework analyzer for summary
        cat_analyzer = CategoricalTargetAnalyzer(min_samples_per_category=10)
        summary_df = cat_analyzer.analyze_multiple(df, categorical_cols, target)

        print("\n📈 Categorical Feature Strength (Cramér's V):")
        print("-" * 60)
        for _, row in summary_df.iterrows():
            if row["cramers_v"] >= 0.3:
                strength = "Strong"
                emoji = "🔴"
            elif row["cramers_v"] >= 0.1:
                strength = "Moderate"
                emoji = "🟡"
            else:
                strength = "Weak"
                emoji = "🟢"
            sig = "***" if row["p_value"] < 0.001 else "**" if row["p_value"] < 0.01 else "*" if row["p_value"] < 0.05 else ""
            print(f"  {emoji} {row['feature']}: V={row['cramers_v']:.3f} ({strength}) {sig}")

        # Detailed analysis for each categorical feature
        for col_name in categorical_cols[:5]:
            result = cat_analyzer.analyze(df, col_name, target)

            print(f"\n{'='*60}")
            print(f"📊 {col_name.upper()}")
            print("="*60)

            # Display stats table
            if len(result.category_stats) > 0:
                display_stats = result.category_stats[['category', 'total_count', 'retention_rate', 'lift', 'pct_of_total']].copy()
                display_stats['retention_rate'] = display_stats['retention_rate'].apply(lambda x: f"{x:.1%}")
                display_stats['lift'] = display_stats['lift'].apply(lambda x: f"{x:.2f}x")
                display_stats['pct_of_total'] = display_stats['pct_of_total'].apply(lambda x: f"{x:.1%}")
                display_stats.columns = [col_name, 'Count', 'Retention Rate', 'Lift', '% of Data']
                display_table(display_stats)

                # Stacked bar chart
                cat_stats = result.category_stats
                categories = cat_stats['category'].tolist()
                retained_counts = cat_stats['retained_count'].tolist()
                churned_counts = cat_stats['churned_count'].tolist()

                fig = go.Figure()

                fig.add_trace(go.Bar(
                    name='Retained',
                    x=categories,
                    y=retained_counts,
                    marker_color='rgba(46, 204, 113, 0.8)',
                    text=[f"{r/(r+c)*100:.0f}%" for r, c in zip(retained_counts, churned_counts)],
                    textposition='inside',
                    textfont=dict(color='white', size=12)
                ))

                fig.add_trace(go.Bar(
                    name='Churned',
                    x=categories,
                    y=churned_counts,
                    marker_color='rgba(231, 76, 60, 0.8)',
                    text=[f"{c/(r+c)*100:.0f}%" for r, c in zip(retained_counts, churned_counts)],
                    textposition='inside',
                    textfont=dict(color='white', size=12)
                ))

                fig.update_layout(
                    barmode='stack',
                    title=f"Retention by {col_name}",
                    xaxis_title=col_name,
                    yaxis_title="Count",
                    template='plotly_white',
                    height=350,
                    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5)
                )
                display_figure(fig)

                # Flag high-risk categories from framework result
                if result.high_risk_categories:
                    print("\n  ⚠️ High-risk categories (lift < 0.9x):")
                    for cat in result.high_risk_categories:
                        cat_row = cat_stats[cat_stats['category'] == cat].iloc[0]
                        print(f"     • {cat}: {cat_row['retention_rate']:.1%} retention ({cat_row['lift']:.2f}x lift)")
    else:
        print("\n  ℹ️ No categorical columns detected.")
else:
    print("No target column available for categorical analysis.")

CATEGORICAL FEATURE ANALYSIS
Overall retention rate: 10.0%

📈 Categorical Feature Strength (Cramér's V):
------------------------------------------------------------
  🟡 cohort_quarter: V=0.191 (Moderate) ***
  🟡 recency_bucket: V=0.159 (Moderate) ***
  🟢 risk_tier: V=0.082 (Weak) ***
  🟢 activity_tier: V=0.068 (Weak) ***
  🟢 lifecycle_quadrant: V=0.017 (Weak) ***

📊 ACTIVITY_TIER


activity_tier,Count,Retention Rate,Lift,% of Data
low,86850,11.8%,1.18x,45.0%
high,29143,11.3%,1.13x,15.1%
mid,77007,7.5%,0.75x,39.9%



  ⚠️ High-risk categories (lift < 0.9x):
     • mid: 7.5% retention (0.75x lift)

📊 RISK_TIER


risk_tier,Count,Retention Rate,Lift,% of Data
low,105957,12.2%,1.22x,54.9%
high,21423,8.1%,0.81x,11.1%
med,65620,7.1%,0.71x,34.0%



  ⚠️ High-risk categories (lift < 0.9x):
     • high: 8.1% retention (0.81x lift)
     • med: 7.1% retention (0.71x lift)

📊 LIFECYCLE_QUADRANT


...


📊 COHORT_QUARTER


cohort_quarter,Count,Retention Rate,Lift,% of Data
2023Q4,12545,16.9%,1.69x,6.5%
2024Q2,14089,16.4%,1.64x,7.3%
2024Q3,17563,15.4%,1.54x,9.1%
2022Q1,1351,14.3%,1.43x,0.7%
2025Q1,16598,14.0%,1.40x,8.6%
2023Q2,10036,13.5%,1.35x,5.2%
2024Q1,11966,12.9%,1.29x,6.2%
2024Q4,12545,12.3%,1.23x,6.5%
2023Q3,12931,11.9%,1.19x,6.7%
2022Q3,7141,10.8%,1.08x,3.7%



  ⚠️ High-risk categories (lift < 0.9x):
     • 2025Q2: 5.6% retention (0.56x lift)
     • 2022Q2: 5.3% retention (0.53x lift)
     • 2025Q3: 2.7% retention (0.27x lift)
     • 2025Q4: 0.0% retention (0.00x lift)
     • 2026Q1: 0.0% retention (0.00x lift)
     • 2026Q2: 0.0% retention (0.00x lift)

📊 RECENCY_BUCKET


recency_bucket,Count,Retention Rate,Lift,% of Data
>180d,157102,12.3%,1.23x,81.4%
0-7d,1351,0.0%,0.00x,0.7%
31-90d,14282,0.0%,0.00x,7.4%
8-30d,3860,0.0%,0.00x,2.0%
91-180d,16405,0.0%,0.00x,8.5%



  ⚠️ High-risk categories (lift < 0.9x):
     • 0-7d: 0.0% retention (0.00x lift)
     • 31-90d: 0.0% retention (0.00x lift)
     • 8-30d: 0.0% retention (0.00x lift)
     • 91-180d: 0.0% retention (0.00x lift)


[//]: # (cr:doc name='5_7_scatter_plot_matrix_sample' id=003bfde3)
## 5.7 Scatter Plot Matrix (Sample)

Visual exploration of pairwise relationships between numeric features.

**📖 How to Read the Scatter Matrix:**
- **Diagonal**: Distribution of each feature (histogram or density)
- **Off-diagonal**: Scatter plot showing relationship between two features
- Each row/column represents one feature

**🔍 What to Look For:**

| Pattern | What It Means | Action |
|---------|--------------|--------|
| **Linear trend** (diagonal line of points) | Strong correlation | Check if redundant; may cause multicollinearity |
| **Curved pattern** | Non-linear relationship | Consider polynomial features or transformations |
| **Clusters/groups** | Natural segments in data | May benefit from segment-aware modeling |
| **Fan shape** (spreading out) | Heteroscedasticity | May need log transform or robust methods |
| **Random scatter** | No relationship | Features are independent |

**⚠️ Cautions:**
- Sample shown (max 1000 points) for performance - patterns may differ in full data
- Look for the same patterns in correlation matrix (section 4.2) to confirm

In [10]:
# @cr:code name='plot_scatter_pairs' id=528ee553
top_numeric = numeric_cols[:4] if len(numeric_cols) > 4 else numeric_cols

if len(top_numeric) >= 2:
    fig = charts.scatter_matrix(
        safe_sample(df[top_numeric], 1000),
        title="Scatter Plot Matrix (Sample)"
    )
    display_figure(fig)

[//]: # (cr:doc name='interpreting_the_scatter_matrix_above' id=d6311565)
### Interpreting the Scatter Matrix Above

**🎯 Key Questions to Answer:**

1. **Are any features redundant?**
   - Look for tight linear patterns → high correlation → consider dropping one
   - Cross-reference with high correlation pairs in section 4.3

2. **Are there natural customer segments?**
   - Distinct clusters suggest different customer types
   - Links to segment-aware outlier analysis in notebook 03

3. **Do relationships suggest feature engineering?**
   - Curved patterns → polynomial or interaction terms may help
   - Ratios between correlated features may be more predictive

4. **Are distributions suitable for linear models?**
   - Fan shapes or heavy skew → consider transformations
   - Outlier clusters → verify with segment analysis

**💡 Pro Tip:** Hover over points in the interactive plot to see exact values. Look for outliers that appear across multiple scatter plots - these may be influential observations worth investigating.

[//]: # (cr:doc name='5_8_datetime_feature_analysis' id=8145b868)
## 5.8 Datetime Feature Analysis

Temporal patterns can reveal important retention signals - when customers joined, their last activity, and seasonal patterns.

**📖 What to Look For:**
- **Cohort effects**: Do customers who joined in certain periods have different retention?
- **Recency patterns**: How does time since last activity relate to retention?
- **Seasonal trends**: Are there monthly or quarterly patterns?

**📊 Common Temporal Features:**
| Feature Type | Example | Typical Insight |
|-------------|---------|-----------------|
| **Tenure** | Days since signup | Longer tenure often = higher retention |
| **Recency** | Days since last order | Recent activity = engaged customer |
| **Cohort** | Signup month/year | Economic conditions affect cohorts |
| **Day of Week** | Signup day | Weekend vs weekday patterns |

In [11]:
# @cr:code name='analyze_temporal_target' id=a5eefead
from customer_retention.stages.profiling import TemporalTargetAnalyzer

datetime_cols = [
    name for name, col in findings.columns.items()
    if col.inferred_type == ColumnType.DATETIME
]

print("=" * 80)
print("DATETIME FEATURE ANALYSIS")
print("=" * 80)
print(f"Detected datetime columns: {datetime_cols}")

if datetime_cols and findings.target_column:
    target = findings.target_column
    overall_retention = df[target].mean()

    # Use framework analyzer
    temporal_analyzer = TemporalTargetAnalyzer(min_samples_per_period=10)

    for col_name in datetime_cols[:3]:
        result = temporal_analyzer.analyze(df, col_name, target)

        print(f"\n{'='*60}")
        print(f"📅 {col_name.upper()}")
        print("="*60)

        if result.n_valid_dates == 0:
            print("  No valid dates found")
            continue

        print(f"  Date range: {result.min_date} to {result.max_date}")
        print(f"  Valid dates: {result.n_valid_dates:,}")

        # 1. Retention by Year (from framework result)
        if len(result.yearly_stats) > 1:
            print(f"\n  📊 Retention by Year: Trend is {result.yearly_trend}")

            year_stats = result.yearly_stats

            fig = make_subplots(rows=1, cols=2, subplot_titles=["Retention Rate by Year", "Customer Count by Year"],
                               column_widths=[0.6, 0.4])

            fig.add_trace(
                go.Scatter(
                    x=year_stats['period'].astype(str),
                    y=year_stats['retention_rate'],
                    mode='lines+markers',
                    name='Retention Rate',
                    line=dict(color='#3498db', width=3),
                    marker=dict(size=10)
                ),
                row=1, col=1
            )
            fig.add_hline(y=overall_retention, line_dash="dash", line_color="gray",
                         annotation_text=f"Overall: {overall_retention:.1%}", row=1, col=1)

            fig.add_trace(
                go.Bar(
                    x=year_stats['period'].astype(str),
                    y=year_stats['count'],
                    name='Count',
                    marker_color='rgba(52, 152, 219, 0.6)'
                ),
                row=1, col=2
            )

            fig.update_layout(height=350, template='plotly_white', showlegend=False)
            fig.update_yaxes(tickformat='.0%', row=1, col=1)
            display_figure(fig)

        # 2. Retention by Month (from framework result)
        if len(result.monthly_stats) > 1:
            print("\n  📊 Retention by Month (Seasonality):")

            month_stats = result.monthly_stats
            colors = ['rgba(46, 204, 113, 0.7)' if r >= overall_retention else 'rgba(231, 76, 60, 0.7)'
                     for r in month_stats['retention_rate']]

            fig = go.Figure()
            fig.add_trace(go.Bar(
                x=month_stats['month_name'],
                y=month_stats['retention_rate'],
                marker_color=colors,
                text=[f"{r:.0%}" for r in month_stats['retention_rate']],
                textposition='outside'
            ))
            fig.add_hline(y=overall_retention, line_dash="dash", line_color="gray",
                         annotation_text=f"Overall: {overall_retention:.1%}")

            fig.update_layout(
                title=f"Monthly Retention Pattern ({col_name})",
                xaxis_title="Month",
                yaxis_title="Retention Rate",
                template='plotly_white',
                height=350,
                yaxis_tickformat='.0%'
            )
            display_figure(fig)

            # Seasonal insights from framework
            if result.seasonal_spread > 0.05:
                print(f"  📈 Seasonal spread: {result.seasonal_spread:.1%}")
                print(f"     Best month: {result.best_month}")
                print(f"     Worst month: {result.worst_month}")

        # 3. Retention by Day of Week (from framework result)
        if len(result.dow_stats) > 1:
            print("\n  📊 Retention by Day of Week:")

            dow_stats = result.dow_stats
            colors = ['rgba(46, 204, 113, 0.7)' if r >= overall_retention else 'rgba(231, 76, 60, 0.7)'
                     for r in dow_stats['retention_rate']]

            fig = go.Figure()
            fig.add_trace(go.Bar(
                x=dow_stats['day_name'],
                y=dow_stats['retention_rate'],
                marker_color=colors,
                text=[f"{r:.0%}" for r in dow_stats['retention_rate']],
                textposition='outside'
            ))
            fig.add_hline(y=overall_retention, line_dash="dash", line_color="gray")

            fig.update_layout(
                title=f"Day of Week Pattern ({col_name})",
                xaxis_title="Day of Week",
                yaxis_title="Retention Rate",
                template='plotly_white',
                height=300,
                yaxis_tickformat='.0%'
            )
            display_figure(fig)
else:
    if not datetime_cols:
        print("\n  ℹ️ No datetime columns detected in this dataset.")
        print("     Consider adding date parsing in notebook 01 if dates exist as strings.")
    else:
        print("\n  ℹ️ No target column available for retention analysis.")

DATETIME FEATURE ANALYSIS
Detected datetime columns: ['churn_end', 'as_of_date']

📅 CHURN_END
  Date range: 2022-09-13 17:49:00+00:00 to 2026-02-04 10:28:00+00:00
  Valid dates: 19,300

  📊 Retention by Year: Trend is stable



  📊 Retention by Month (Seasonality):



  📊 Retention by Day of Week:



📅 AS_OF_DATE
  Date range: 2022-11-07 00:00:00 to 2026-07-13 00:00:00
  Valid dates: 193,000

  📊 Retention by Year: Trend is stable



  📊 Retention by Month (Seasonality):


[//]: # (cr:doc name='5_9_actionable_recommendations_summary' id=8ff2c92e)
## 5.9 Actionable Recommendations Summary

This section consolidates all relationship analysis findings into **actionable recommendations** organized by their impact on the modeling pipeline.

**📋 Recommendation Categories:**

| Category | Purpose | Impact |
|----------|---------|--------|
| **Feature Selection** | Which features to keep/drop | Reduces noise, improves interpretability |
| **Feature Engineering** | New features to create | Captures interactions, improves accuracy |
| **Stratification** | Train/test split strategy | Ensures fair evaluation, prevents leakage |
| **Model Selection** | Which algorithms to try | Matches model to data characteristics |

In [12]:
# @cr:code name='generate_recommendations' id=ce9806ee
# Generate comprehensive actionable recommendations
recommender = RelationshipRecommender()

# Gather columns by type
numeric_features = [
    name for name, col in findings.columns.items()
    if col.inferred_type in [ColumnType.NUMERIC_CONTINUOUS, ColumnType.NUMERIC_DISCRETE]
    and name != findings.target_column
    and name not in TEMPORAL_METADATA_COLS
]
categorical_features = [
    name for name, col in findings.columns.items()
    if col.inferred_type in [ColumnType.CATEGORICAL_NOMINAL, ColumnType.CATEGORICAL_ORDINAL]
    and name not in TEMPORAL_METADATA_COLS
]

# Run comprehensive analysis — reuse precomputed correlation matrix and effect sizes
analysis_summary = recommender.analyze(
    df,
    numeric_cols=numeric_features,
    categorical_cols=categorical_features,
    target_col=findings.target_column,
    correlation_matrix=corr_matrix,
    effect_sizes=_effect_sizes_result.effect_sizes if _effect_sizes_result else None,
)

print("=" * 80)
print("ACTIONABLE RECOMMENDATIONS FROM RELATIONSHIP ANALYSIS")
print("=" * 80)

# Group recommendations by category
grouped_recs = analysis_summary.recommendations_by_category
high_priority = analysis_summary.high_priority_actions

if high_priority:
    print(f"\n🔴 HIGH PRIORITY ACTIONS ({len(high_priority)}):")
    print("-" * 60)
    for rec in high_priority:
        print(f"\n  📌 {rec.title}")
        print(f"     {rec.description}")
        print(f"     → Action: {rec.action}")
        if rec.affected_features:
            print(f"     → Features: {', '.join(rec.affected_features[:5])}")

# Persist recommendations to registry
for pair in analysis_summary.multicollinear_pairs:
    registry.add_gold_drop_multicollinear(
        column=pair["feature1"], correlated_with=pair["feature2"],
        correlation=pair["correlation"],
        rationale=f"High correlation ({pair['correlation']:.2f}) - consider dropping one",
        source_notebook="05_relationship_analysis"
    )

for predictor in analysis_summary.strong_predictors:
    registry.add_gold_prioritize_feature(
        column=predictor["feature"], effect_size=predictor["effect_size"],
        correlation=predictor["correlation"],
        rationale=f"Strong predictor with effect size {predictor['effect_size']:.2f}",
        source_notebook="05_relationship_analysis"
    )

for weak_col in analysis_summary.weak_predictors[:10]:
    registry.add_gold_drop_weak(
        column=weak_col, effect_size=0.0, correlation=0.0,
        rationale="Negligible predictive power",
        source_notebook="05_relationship_analysis"
    )

# Persist ratio feature recommendations
for rec in grouped_recs.get(RecommendationCategory.FEATURE_ENGINEERING, []):
    if "ratio" in rec.title.lower():
        for col1, col2, _corr in rec.evidence.get("moderate_pairs", []):
            registry.add_silver_ratio(
                column=f"{col1}_to_{col2}_ratio",
                numerator=col1, denominator=col2,
                rationale=rec.description, source_notebook="05_relationship_analysis"
            )
    elif "interaction" in rec.title.lower() and len(rec.affected_features) >= 2:
        for i, f1 in enumerate(rec.affected_features[:3]):
            for f2 in rec.affected_features[i+1:4]:
                registry.add_silver_interaction(
                    column=f"{f1}_x_{f2}", features=[f1, f2],
                    rationale=rec.description, source_notebook="05_relationship_analysis"
                )

# Store for findings metadata
findings.metadata["relationship_analysis"] = {
    "n_recommendations": len(analysis_summary.recommendations),
    "n_high_priority": len(high_priority),
    "strong_predictors": [p["feature"] for p in analysis_summary.strong_predictors],
    "weak_predictors": analysis_summary.weak_predictors[:5],
    "multicollinear_pairs": [(p["feature1"], p["feature2"]) for p in analysis_summary.multicollinear_pairs],
}

print(f"\n✅ Persisted {len(analysis_summary.multicollinear_pairs)} multicollinearity recommendations")
print(f"✅ Persisted {len(analysis_summary.strong_predictors)} strong predictor recommendations")
print(f"✅ Persisted {min(len(analysis_summary.weak_predictors), 10)} weak predictor recommendations")


ACTIONABLE RECOMMENDATIONS FROM RELATIONSHIP ANALYSIS

🔴 HIGH PRIORITY ACTIONS (2665):
------------------------------------------------------------

  📌 Remove multicollinear feature
     event_count_7d and event_count_14d are highly correlated (r=1.00)
     → Action: Consider dropping one of these features. Keep the one with stronger business meaning or higher target correlation.
     → Features: event_count_7d, event_count_14d

  📌 Remove multicollinear feature
     event_count_7d and event_count_30d are highly correlated (r=0.97)
     → Action: Consider dropping one of these features. Keep the one with stronger business meaning or higher target correlation.
     → Features: event_count_7d, event_count_30d

  📌 Remove multicollinear feature
     event_count_7d and edi_transaction_type_sum_7d are highly correlated (r=1.00)
     → Action: Consider dropping one of these features. Keep the one with stronger business meaning or higher target correlation.
     → Features: event_count_7d, e

[//]: # (cr:doc name='5_9_1_feature_selection_recommendations' id=19c6dcf4)
### 5.9.1 Feature Selection Recommendations

**What these recommendations tell you:**
- Which features to **prioritize** (strong predictors)
- Which features to **consider dropping** (weak predictors, redundant features)
- Which feature pairs cause **multicollinearity** issues

**📊 Decision Guide:**

| Finding | Linear Models | Tree-Based Models |
|---------|--------------|-------------------|
| Strong predictors | Include - will have high coefficients | Include - will appear early in splits |
| Weak predictors | Consider dropping | May help in interactions |
| Multicollinear pairs | Drop one feature | Can keep both (trees handle it) |

In [13]:
# @cr:code name='display_feature_selection' id=96df0793
# Feature Selection Recommendations
selection_recs = grouped_recs.get(RecommendationCategory.FEATURE_SELECTION, [])

print("=" * 70)
print("FEATURE SELECTION")
print("=" * 70)

# Strong predictors summary
if analysis_summary.strong_predictors:
    print("\n✅ STRONG PREDICTORS (prioritize these):")
    strong_df = native_pd.DataFrame(analysis_summary.strong_predictors)
    strong_df["effect_size"] = strong_df["effect_size"].apply(lambda x: f"{x:+.3f}")
    strong_df["correlation"] = strong_df["correlation"].apply(lambda x: f"{x:+.3f}")
    strong_df = strong_df.sort_values("effect_size", key=lambda x: x.str.replace("+", "").astype(float).abs(), ascending=False)
    display_table(strong_df)

    print("\n   💡 These features show strong discrimination between retained/churned customers.")
    print("   → Ensure they're included in your model")
    print("   → Check for data quality issues that could inflate their importance")

# Weak predictors summary
if analysis_summary.weak_predictors:
    print(f"\n⚪ WEAK PREDICTORS (consider dropping): {', '.join(analysis_summary.weak_predictors[:5])}")
    print("   → Low individual predictive power, but may help in combination")

# Multicollinearity summary
if analysis_summary.multicollinear_pairs:
    print("\n⚠️ MULTICOLLINEAR PAIRS (drop one from each pair for linear models):")
    for pair in analysis_summary.multicollinear_pairs:
        print(f"   • {pair['feature1']} ↔ {pair['feature2']}: r = {pair['correlation']:.2f}")
    print("\n   💡 For each pair, keep the feature with:")
    print("      - Stronger business meaning")
    print("      - Higher target correlation")
    print("      - Fewer missing values")

# Display all feature selection recommendations
if selection_recs:
    print("\n" + "-" * 70)
    print("DETAILED RECOMMENDATIONS:")
    for rec in selection_recs:
        priority_icon = "🔴" if rec.priority == "high" else "🟡" if rec.priority == "medium" else "🟢"
        print(f"\n{priority_icon} {rec.title}")
        print(f"   {rec.description}")
        print(f"   → {rec.action}")

FEATURE SELECTION

✅ STRONG PREDICTORS (prioritize these):


feature,correlation,effect_size
lag3_resolved_at_delta_hours_mean,+0.442,+1.406
lag3_time_to_close_hours_mean,+0.442,+1.406
lag3_resolved_at_delta_hours_max,+0.430,+1.361
lag3_time_to_close_hours_max,+0.430,+1.361
lag3_resolved_at_delta_hours_sum,+0.347,+0.992
lag3_time_to_close_hours_sum,+0.347,+0.992
satisfaction_mean_all_time,-0.298,-0.932
satisfaction_max_all_time,-0.276,-0.855
lag0_satisfaction_max,-0.269,-0.824
lag0_satisfaction_mean,-0.266,-0.817



   💡 These features show strong discrimination between retained/churned customers.
   → Ensure they're included in your model
   → Check for data quality issues that could inflate their importance

⚪ WEAK PREDICTORS (consider dropping): event_count_7d, event_count_14d, event_count_30d, event_count_90d, event_count_all_time
   → Low individual predictive power, but may help in combination

⚠️ MULTICOLLINEAR PAIRS (drop one from each pair for linear models):
   • event_count_7d ↔ event_count_14d: r = 1.00
   • event_count_7d ↔ event_count_30d: r = 0.97
   • event_count_7d ↔ edi_transaction_type_sum_7d: r = 1.00
   • event_count_7d ↔ edi_transaction_type_count_7d: r = 1.00
   • event_count_7d ↔ amount_sum_7d: r = 0.99
   • event_count_7d ↔ amount_max_7d: r = 0.96
   • event_count_7d ↔ amount_count_7d: r = 1.00
   • event_count_7d ↔ file_size_kb_sum_7d: r = 0.99
   • event_count_7d ↔ file_size_kb_max_7d: r = 0.72
   • event_count_7d ↔ file_size_kb_count_7d: r = 1.00
   • event_count_7d ↔ 

[//]: # (cr:doc name='5_9_2_stratification_recommendations' id=550ab7af)
### 5.9.2 Stratification Recommendations

**What these recommendations tell you:**
- How to **split your data** for training and testing
- Which **segments require special attention** in sampling
- **High-risk segments** that need adequate representation

**⚠️ Why This Matters:**
- Random splits can under-represent rare segments
- High-risk segments may be systematically excluded
- Model evaluation will be biased without proper stratification

**📊 Implementation:**
```python
from sklearn.model_selection import train_test_split

# Stratified split by target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Multi-column stratification (for categorical segments)
df['stratify_col'] = df['target'].astype(str) + '_' + df['segment']
```

In [14]:
# @cr:code name='display_stratification' id=529bd986
# Stratification Recommendations
strat_recs = grouped_recs.get(RecommendationCategory.STRATIFICATION, [])

print("=" * 70)
print("STRATIFICATION (Train/Test Split Strategy)")
print("=" * 70)

# High-risk segments
if analysis_summary.high_risk_segments:
    print("\n🎯 HIGH-RISK SEGMENTS (ensure representation in training data):")
    risk_df = native_pd.DataFrame(analysis_summary.high_risk_segments)
    risk_df["retention_rate"] = risk_df["retention_rate"].apply(lambda x: f"{x:.1%}")
    risk_df["lift"] = risk_df["lift"].apply(lambda x: f"{x:.2f}x")
    display_table(risk_df[["feature", "segment", "count", "retention_rate", "lift"]])

    print("\n   💡 These segments have below-average retention.")
    print("   → Ensure they're adequately represented in both train and test sets")
    print("   → Consider oversampling or class weights in modeling")

# Display all stratification recommendations
if strat_recs:
    print("\n" + "-" * 70)
    print("STRATIFICATION RECOMMENDATIONS:")
    for rec in strat_recs:
        priority_icon = "🔴" if rec.priority == "high" else "🟡" if rec.priority == "medium" else "🟢"
        print(f"\n{priority_icon} {rec.title}")
        print(f"   {rec.description}")
        print(f"   → {rec.action}")
else:
    print("\n✅ No special stratification requirements detected.")
    print("   Standard stratified split by target variable is sufficient.")

STRATIFICATION (Train/Test Split Strategy)

🎯 HIGH-RISK SEGMENTS (ensure representation in training data):


feature,segment,count,retention_rate,lift
activity_tier,mid,77007,7.5%,0.75x
risk_tier,high,21423,8.1%,0.81x
risk_tier,med,65620,7.1%,0.71x
cohort_quarter,2022Q2,3667,5.3%,0.53x
cohort_quarter,2025Q2,13896,5.6%,0.56x
cohort_quarter,2025Q3,14475,2.7%,0.27x
cohort_quarter,2025Q4,19493,0.0%,0.00x
cohort_quarter,2026Q1,8878,0.0%,0.00x
cohort_quarter,2026Q2,386,0.0%,0.00x
recency_bucket,0-7d,1351,0.0%,0.00x



   💡 These segments have below-average retention.
   → Ensure they're adequately represented in both train and test sets
   → Consider oversampling or class weights in modeling

----------------------------------------------------------------------
STRATIFICATION RECOMMENDATIONS:

🟡 Stratify by cohort_quarter
   Significant variation in retention rates across cohort_quarter categories (spread: 16.9%)
   → Use stratified sampling by cohort_quarter in train/test split to ensure all segments are represented.

🟡 Stratify by recency_bucket
   Significant variation in retention rates across recency_bucket categories (spread: 12.3%)
   → Use stratified sampling by recency_bucket in train/test split to ensure all segments are represented.

🔴 Monitor high-risk segments
   Segments with below-average retention: mid, med, high
   → Target these segments for intervention campaigns and ensure adequate representation in training data.


[//]: # (cr:doc name='5_9_3_model_selection_recommendations' id=f602b254)
### 5.9.3 Model Selection Recommendations

**What these recommendations tell you:**
- Which **model types** are well-suited for your data characteristics
- **Linear vs non-linear** based on relationship patterns
- **Ensemble considerations** based on feature interactions

**📊 Model Selection Guide Based on Data Characteristics:**

| Data Characteristic | Recommended Models | Reason |
|---------------------|-------------------|--------|
| Strong linear relationships | Logistic Regression, Linear SVM | Interpretable, fast, less overfit risk |
| Non-linear patterns | Random Forest, XGBoost, LightGBM | Capture complex interactions |
| High multicollinearity | Tree-based models | Robust to correlated features |
| Many categorical features | CatBoost, LightGBM | Native categorical handling |
| Imbalanced classes | Any with class_weight='balanced' | Adjust for minority class |

In [15]:
# @cr:code name='display_model_selection' id=5b7c3bff
# Model Selection Recommendations
model_recs = grouped_recs.get(RecommendationCategory.MODEL_SELECTION, [])

print("=" * 70)
print("MODEL SELECTION")
print("=" * 70)

if model_recs:
    for rec in model_recs:
        priority_icon = "🔴" if rec.priority == "high" else "🟡" if rec.priority == "medium" else "🟢"
        print(f"\n{priority_icon} {rec.title}")
        print(f"   {rec.description}")
        print(f"   → {rec.action}")

# Summary recommendations based on data characteristics
print("\n" + "-" * 70)
print("RECOMMENDED MODELING APPROACH:")

has_multicollinearity = len(analysis_summary.multicollinear_pairs) > 0
has_strong_linear = len([p for p in analysis_summary.strong_predictors if abs(p.get("effect_size", 0)) >= 0.5]) > 0
has_categoricals = len(categorical_features) > 0

if has_strong_linear and not has_multicollinearity:
    print("\n✅ RECOMMENDED: Start with Logistic Regression")
    print("   • Strong linear relationships detected")
    print("   • Interpretable coefficients for business insights")
    print("   • Fast training and inference")
    print("   • Then compare with tree-based ensemble for potential improvement")
elif has_multicollinearity:
    print("\n✅ RECOMMENDED: Start with Random Forest or XGBoost")
    print("   • Multicollinearity present - tree models handle it naturally")
    print("   • Can keep all features without VIF analysis")
    print("   • Use feature importance to understand contributions")
else:
    print("\n✅ RECOMMENDED: Compare Linear and Tree-Based Models")
    print("   • No clear linear dominance - test both approaches")
    print("   • Logistic Regression for interpretability baseline")
    print("   • Random Forest/XGBoost for potential accuracy gain")

if has_categoricals:
    print("\n💡 CATEGORICAL HANDLING:")
    print("   • For tree models: Consider CatBoost or LightGBM with native categorical support")
    print("   • For linear models: Use target encoding for high-cardinality features")

MODEL SELECTION

🟡 Consider tree-based models for multicollinearity
   Found 4663 highly correlated feature pairs
   → Tree-based models (Random Forest, XGBoost) are robust to multicollinearity. For linear models, remove redundant features first.

🟡 Linear models may perform well
   Strong linear relationships detected (avg effect size: 0.88)
   → Start with Logistic Regression as baseline. Clear feature-target relationships suggest interpretable models may work well.

----------------------------------------------------------------------
RECOMMENDED MODELING APPROACH:

✅ RECOMMENDED: Start with Random Forest or XGBoost
   • Multicollinearity present - tree models handle it naturally
   • Can keep all features without VIF analysis
   • Use feature importance to understand contributions

💡 CATEGORICAL HANDLING:
   • For tree models: Consider CatBoost or LightGBM with native categorical support
   • For linear models: Use target encoding for high-cardinality features


[//]: # (cr:doc name='5_9_4_feature_engineering_recommendations' id=6de9d98c)
### 5.9.4 Feature Engineering Recommendations

**What these recommendations tell you:**
- **Interaction features** to create based on correlation patterns
- **Ratio features** that may capture relative relationships
- **Polynomial features** for non-linear patterns

**📊 Common Feature Engineering Patterns:**

| Pattern Found | Feature to Create | Example |
|---------------|------------------|---------|
| Moderate correlation | Ratio feature | `feature_a / feature_b` |
| Both features predictive | Interaction term | `feature_a * feature_b` |
| Curved scatter pattern | Polynomial | `feature_a ** 2` |
| Related semantics | Difference | `total_orders - returned_orders` |

In [16]:
# @cr:code name='display_feature_engineering' id=89f87163
# Feature Engineering Recommendations
eng_recs = grouped_recs.get(RecommendationCategory.FEATURE_ENGINEERING, [])

print("=" * 70)
print("FEATURE ENGINEERING")
print("=" * 70)

if eng_recs:
    for rec in eng_recs:
        priority_icon = "🔴" if rec.priority == "high" else "🟡" if rec.priority == "medium" else "🟢"
        print(f"\n{priority_icon} {rec.title}")
        print(f"   {rec.description}")
        print(f"   → {rec.action}")
        if rec.affected_features:
            print(f"   → Features: {', '.join(rec.affected_features[:5])}")
else:
    print("\n✅ No specific feature engineering recommendations based on correlation patterns.")
    print("   Consider domain-specific features based on business knowledge.")

# Additional suggestions based on strong predictors
if analysis_summary.strong_predictors:
    print("\n" + "-" * 70)
    print("POTENTIAL INTERACTION FEATURES:")
    strong_features = [p["feature"] for p in analysis_summary.strong_predictors[:5]]
    if len(strong_features) >= 2:
        print("\n   Based on strong predictors, consider interactions between:")
        for i, f1 in enumerate(strong_features[:3]):
            for f2 in strong_features[i+1:4]:
                print(f"   • {f1} × {f2}")
        print("\n   💡 Tree-based models discover interactions automatically.")
        print("   → For linear models, create explicit interaction columns.")

FEATURE ENGINEERING

🟢 Consider ratio features
   Moderately correlated pairs may benefit from ratio features: event_count_7d/event_count_90d, event_count_7d/event_count_180d, event_count_7d/edi_transaction_type_max_7d
   → Create ratio features (e.g., feature_a / feature_b) to capture relative relationships.
   → Features: event_count_7d, event_count_7d, event_count_7d, event_count_90d, event_count_180d

🟢 Test feature interactions
   Interaction terms may capture non-linear relationships
   → Use PolynomialFeatures(interaction_only=True) or tree-based models which automatically discover interactions.
   → Features: event_count_7d, event_count_14d, event_count_30d, event_count_90d

----------------------------------------------------------------------
POTENTIAL INTERACTION FEATURES:

   Based on strong predictors, consider interactions between:
   • satisfaction_mean_all_time × satisfaction_max_all_time
   • satisfaction_mean_all_time × lag0_satisfaction_mean
   • satisfaction_mean_al

[//]: # (cr:doc name='5_9_5_recommendations_summary_table' id=baa8f741)
### 5.9.5 Recommendations Summary Table

In [17]:
# @cr:code name='display_recommendations_summary' id=c1c8995a
# Create summary table of all recommendations
all_recs_data = []
for rec in analysis_summary.recommendations:
    all_recs_data.append({
        "Category": rec.category.value.replace("_", " ").title(),
        "Priority": rec.priority.upper(),
        "Recommendation": rec.title,
        "Action": rec.action[:80] + "..." if len(rec.action) > 80 else rec.action
    })

if all_recs_data:
    recs_df = native_pd.DataFrame(all_recs_data)

    # Sort by priority
    priority_order = {"HIGH": 0, "MEDIUM": 1, "LOW": 2}
    recs_df["_sort"] = recs_df["Priority"].map(priority_order)
    recs_df = recs_df.sort_values("_sort").drop("_sort", axis=1)

    print("=" * 80)
    print("ALL RECOMMENDATIONS SUMMARY")
    print("=" * 80)
    print(f"\nTotal: {len(recs_df)} recommendations")
    print(f"  \U0001f534 High priority: {len(recs_df[recs_df['Priority'] == 'HIGH'])}")
    print(f"  \U0001f7e1 Medium priority: {len(recs_df[recs_df['Priority'] == 'MEDIUM'])}")
    print(f"  \U0001f7e2 Low priority: {len(recs_df[recs_df['Priority'] == 'LOW'])}")

    display_table(recs_df)

# Save updated findings and recommendations registry
findings.save(FINDINGS_PATH)
registry.save(RECOMMENDATIONS_PATH)

print(f"\n\u2705 Findings updated with relationship analysis: {FINDINGS_PATH}")
print(f"\u2705 Recommendations registry saved: {RECOMMENDATIONS_PATH}")
print(f"   Total recommendations in registry: {len(registry.all_recommendations)}")

if _namespace:
    from customer_retention.analysis.auto_explorer.project_context import ProjectContext

    _namespace.merged_dir.mkdir(parents=True, exist_ok=True)
    _all_findings = _namespace.discover_all_findings(prefer_aggregated=True)
    _mgr = ExplorationManager(_namespace.merged_dir, findings_paths=_all_findings)

    _scaffold = []
    _ctx = None
    if _namespace.project_context_path.exists():
        _ctx = ProjectContext.load(_namespace.project_context_path)
        _scaffold = _ctx.merge_scaffold

    _multi = _mgr.create_multi_dataset_findings(merge_scaffold=_scaffold)
    if _ctx:
        for _name, _entry in _ctx.datasets.items():
            if _name in _multi.datasets:
                _multi.datasets[_name].raw_source_path = _entry.path
    _multi.save(str(_namespace.multi_dataset_findings_path))
    print(f"\n\u2705 Merged findings saved to {_namespace.merged_dir}")

ALL RECOMMENDATIONS SUMMARY

Total: 4672 recommendations
  🔴 High priority: 2665
  🟡 Medium priority: 2004
  🟢 Low priority: 3


Category,Priority,Recommendation,Action
Feature Selection,HIGH,Remove multicollinear feature,Consider dropping one of these features. Keep the one with stronger business mea...
Feature Selection,HIGH,Remove multicollinear feature,Consider dropping one of these features. Keep the one with stronger business mea...
Feature Selection,HIGH,Remove multicollinear feature,Consider dropping one of these features. Keep the one with stronger business mea...
Feature Selection,HIGH,Remove multicollinear feature,Consider dropping one of these features. Keep the one with stronger business mea...
Feature Selection,HIGH,Remove multicollinear feature,Consider dropping one of these features. Keep the one with stronger business mea...
Feature Selection,HIGH,Remove multicollinear feature,Consider dropping one of these features. Keep the one with stronger business mea...
Feature Selection,HIGH,Remove multicollinear feature,Consider dropping one of these features. Keep the one with stronger business mea...
Feature Selection,HIGH,Remove multicollinear feature,Consider dropping one of these features. Keep the one with stronger business mea...
Feature Selection,HIGH,Remove multicollinear feature,Consider dropping one of these features. Keep the one with stronger business mea...
Feature Selection,HIGH,Remove multicollinear feature,Consider dropping one of these features. Keep the one with stronger business mea...



✅ Findings updated with relationship analysis: /Users/Vital/python/CustomerRetention/experiments/runs/3set-1e30406f/merged/silver_merged_findings.yaml
✅ Recommendations registry saved: /Users/Vital/python/CustomerRetention/experiments/runs/3set-1e30406f/merged/recommendations.yaml
   Total recommendations in registry: 5152



✅ Merged findings saved to /Users/Vital/python/CustomerRetention/experiments/runs/3set-1e30406f/merged


[//]: # (cr:doc name='summary_what_we_learned' id=ac445fe4)
---

## Summary: What We Learned

In this notebook, we analyzed feature relationships and generated **actionable recommendations** for modeling.

### Analysis Performed

**Numeric Features:**
1. **Correlation Matrix** - Identified multicollinearity issues between feature pairs
2. **Effect Sizes (Cohen's d)** - Quantified how well features discriminate retained vs churned
3. **Box Plots** - Visualized distribution differences between classes
4. **Feature-Target Correlations** - Ranked features by predictive power

**Categorical Features:**
5. **Cramér's V** - Measured association strength for categorical variables
6. **Retention by Category** - Identified high-risk segments
7. **Lift Analysis** - Found categories performing above/below average

**Datetime Features:**
8. **Cohort Analysis** - Retention trends by signup year
9. **Seasonality** - Monthly patterns in retention

### Actionable Recommendations Generated

| Category | What It Tells You | Impact on Pipeline |
|----------|-------------------|-------------------|
| **Feature Selection** | Which features to prioritize/drop | Reduces noise, improves interpretability |
| **Stratification** | How to split train/test | Ensures fair evaluation |
| **Model Selection** | Which algorithms to try first | Matches model to data |
| **Feature Engineering** | Interactions to create | Captures non-linear patterns |

### Key Metrics Reference

| Data Type | Effect Measure | Strong Signal |
|-----------|---------------|---------------|
| Numeric | Cohen's d | \|d\| ≥ 0.8 |
| Numeric | Correlation | \|r\| ≥ 0.5 |
| Categorical | Cramér's V | V ≥ 0.3 |
| Categorical | Lift | < 0.9x or > 1.1x |

---

## Recommended Actions Checklist

Based on the analysis above, here are the key actions to take:

- [ ] **Feature Selection**: Review strong/weak predictors and multicollinear pairs
- [ ] **Stratification**: Use stratified sampling with identified high-risk segments
- [ ] **Model Selection**: Start with recommended model type based on data characteristics
- [ ] **Feature Engineering**: Create interaction features between strong predictors

---

## Next Steps

Continue to **05_feature_opportunities.ipynb** to:
- Generate derived features (tenure, recency, engagement scores)
- Identify interaction features based on relationships found here
- Create business-relevant composite scores
- Review automated feature recommendations

[//]: # (cr:doc name='section' id=8d492aee)
> **Save Reminder:** Save this notebook (Ctrl+S / Cmd+S) before running the next one.
> The next notebook will automatically export this notebook's HTML documentation from the saved file.